# 04 LangChain组件与消息格式

**用途：** 确认项目实际使用的LangChain模块、StructuredTool、Prompt和消息处理边界。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


## 1. 项目里具体用了什么

- `Document`、`RecursiveCharacterTextSplitter`
- `HuggingFaceEmbeddings`、`Chroma`、自定义Scored Retriever
- `ChatPromptTemplate`、`ChatOpenAI`、`StrOutputParser`
- `StructuredTool`和Pydantic args schema
- `with_structured_output(AgentParsePlan)`
- 视觉链的`HumanMessage`

LangGraph是主控，LangChain放在节点内部，不用通用Agent循环替代显式业务图。

In [2]:
from langchain_tools import get_langchain_tool_map
tool_map = get_langchain_tool_map()
tool_rows = []
for name, tool in tool_map.items():
    schema = tool.args_schema.model_json_schema()
    tool_rows.append({
        "工具": name,
        "Schema": tool.args_schema.__name__,
        "字段": ", ".join(schema.get("properties", {}).keys()),
    })
show_table(tool_rows)
check_equal("StructuredTool数量", len(tool_map), 5)

,工具,Schema,字段
0,inventory_tool,InventoryToolArgs,"brand, machine_model, part_name, quality_level"
1,quote_tool,QuoteToolArgs,"brand, machine_model, part_name, quality_level..."
2,logistics_tool,LogisticsToolArgs,"city, part_name, urgent"
3,ticket_tool,TicketToolArgs,"order_id, raw_question"
4,knowledge_tool,KnowledgeToolArgs,"question, top_k"


[PASS] StructuredTool数量 | actual=5, expected=5


{'检查项': 'StructuredTool数量', '状态': 'PASS', '说明': 'actual=5, expected=5'}

In [3]:
from context_manager import make_message, compact_messages, ContextPolicy

messages = [
    make_message("user", "PC200液压泵有没有货？", turn_index=1, request_id="r1"),
    make_message("assistant", "请确认品质。", turn_index=1, request_id="r1"),
    make_message("user", "原厂。", turn_index=2, request_id="r2"),
    make_message("assistant", "请确认数量。", turn_index=2, request_id="r2"),
    make_message("user", "1件。", turn_index=3, request_id="r3"),
]
recent, summary, dropped = compact_messages(
    messages,
    "",
    ContextPolicy(max_recent_messages=4, max_summary_chars=300),
)
show_table(recent)
print("summary:", summary)
check_equal("近期消息保留4条", len(recent), 4)
check_equal("较早消息压缩1条", dropped, 1)

,role,content,turn_index,request_id,created_at
0,assistant,请确认品质。,1,r1,2026-07-28T06:33:55+00:00
1,user,原厂。,2,r2,2026-07-28T06:33:55+00:00
2,assistant,请确认数量。,2,r2,2026-07-28T06:33:55+00:00
3,user,1件。,3,r3,2026-07-28T06:33:55+00:00


summary: 客户: PC200液压泵有没有货？
[PASS] 近期消息保留4条 | actual=4, expected=4
[PASS] 较早消息压缩1条 | actual=1, expected=1


{'检查项': '较早消息压缩1条', '状态': 'PASS', '说明': 'actual=1, expected=1'}

In [4]:
langchain_tests = run_unittest(
    ["tests.test_langchain_integration"],
    project2_root=PROJECT2_ROOT,
)
check("LangChain/RAG集成7条通过", "Ran 7 tests" in langchain_tests.output and "OK" in langchain_tests.output)

$ D:\new things\项目1\day1\.venv\Scripts\python.exe -m unittest tests.test_langchain_integration -v
test_dispatcher_executes_the_canonical_structured_tool (tests.test_langchain_integration.LangChainIntegrationTests.test_dispatcher_executes_the_canonical_structured_tool) ... ok
test_existing_functions_are_exposed_as_structured_tools (tests.test_langchain_integration.LangChainIntegrationTests.test_existing_functions_are_exposed_as_structured_tools) ... ok
test_graph_adds_knowledge_tool_only_when_enabled (tests.test_langchain_integration.LangChainIntegrationTests.test_graph_adds_knowledge_tool_only_when_enabled) ... ok
test_hybrid_keeps_high_confidence_rule_result (tests.test_langchain_integration.LangChainIntegrationTests.test_hybrid_keeps_high_confidence_rule_result) ... ok
test_knowledge_tool_serializes_rag_sources (tests.test_langchain_integration.LangChainIntegrationTests.test_knowledge_tool_serializes_rag_sources) ... ok
test_langchain_failure_falls_back_to_rules (tests.test_langchain

{'检查项': 'LangChain/RAG集成7条通过', '状态': 'PASS', '说明': ''}

## 问题与答案

- **LangChain好处和代价？** 好处是模型、Prompt、Parser、Tool和Retriever协议统一；代价是抽象层、依赖版本和Trace理解成本。
- **RAG能否拆成loader/splitter/embedding/retriever/prompt/LLM chain？** 已经按这些组件拆分，业务入口保持稳定。
- **换Embedding或Vector DB怎么改？** 在`rag_components.py`工厂层替换；重建索引并重测Top-K和距离阈值。
- **有没有callbacks/tracing/LangSmith？** 当前主要是本地execution trace、CSV/JSONL和离线评测；LangSmith尚未正式接入。
- **消息是否已处理？** 已归一化role、截断、摘要、分区并转成受控Prompt。State里仍是自定义字典，不是全量`BaseMessage`。

### 面试代码追问

1. 为什么State不用全部保存`HumanMessage/AIMessage/ToolMessage`？
2. 什么时候需要补`tool_call_id`映射？
3. 为什么StructuredTool只能描述协议，不能决定是否执行？
4. LangChain Agent和LangGraph业务图有什么区别？

### 参考答案

1. **为什么State不全用BaseMessage？** 当前自定义字典更容易序列化进SQLite、记录`turn_index/request_id`并执行截断和摘要，也让业务State不被某个模型消息类绑定。真正调用模型时，再由适配层转换为Prompt或`HumanMessage`。
2. **什么时候需要`tool_call_id`？** 当使用模型原生Tool Calling并把`AIMessage.tool_calls`与多个`ToolMessage`逐一对应时必须保存稳定ID，否则并行工具结果无法正确回填。当前图自己调度StructuredTool，主要靠`request_id + tool_name + arguments`幂等键关联。
3. **StructuredTool为什么不能决定执行？** 它只描述名称、说明、参数Schema和调用函数。是否允许报价、是否缺字段、是否需要人工审批属于业务策略，必须由LangGraph节点和条件边控制，不能交给模型描述层。
4. **LangChain Agent和LangGraph有什么区别？** 通用Agent通常让模型循环选择工具，适合开放任务；本项目涉及报价、售后、恢复和人工介入，需要显式、可测试的状态机。LangChain在节点内部提供组件，LangGraph负责整个业务生命周期。

**代码落点：** `context_manager.py::make_message`、`langchain_tools.py`、`tool_dispatcher.py`、`agent_graph.py`。